# V1: Person Detection & Tracking from Scratch
## Background Subtraction (MOG2) + Centroid/IoU Tracker

This notebook demonstrates the **from-scratch** person tracking pipeline:
- **Detection**: MOG2 Gaussian Mixture background subtraction + contour filtering
- **Tracking**: Custom centroid / IoU tracker with Hungarian assignment (pure NumPy)
- **Deployment**: Designed to run on [Modal.com](https://modal.com) for cloud execution


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "opencv-python-headless", "numpy", "matplotlib", "scipy", "modal", "--quiet"],
    capture_output=True)
print("Dependencies ready")


## 1. Architecture

```
Input Frame
    │
    ▼
[MOG2 Background Subtractor]   ← adaptive per-pixel GMM
    │  foreground mask
    ▼
[Morphological Open + Close]   ← noise removal
    │  clean binary mask
    ▼
[Contour Detection]            ← cv2.findContours
    │  candidate bboxes
    ▼
[Area & Aspect-Ratio Filter]   ← 1500–80000 px², ratio 0.2–4.0
    │  detections
    ▼
[CentroidTracker.update()]     ← Hungarian assign on IoU cost matrix
    │  tracked objects with stable IDs
    ▼
Annotated Output Frame
```


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

from src.v1_scratch.detector import MOG2Detector
from src.v1_scratch.tracker import CentroidTracker
from src.utils import draw_tracks, write_output_video, benchmark_speed, compute_iou

print("All V1 modules imported successfully")


In [ ]:
# ── Generate synthetic test video (150 frames, 640×480, 30 FPS) ──────────────
WIDTH, HEIGHT, N_FRAMES, FPS = 640, 480, 150, 30.0

objects = [
    {"x": 50,  "y": 140, "w": 40, "h": 110, "vx": 4,  "vy": 0, "color": (50, 50, 220)},
    {"x": 500, "y": 190, "w": 38, "h": 100, "vx": -3, "vy": 1, "color": (50, 200, 50)},
    {"x": 280, "y": 80,  "w": 44, "h": 120, "vx": 2,  "vy": 2, "color": (220, 50, 50)},
]
synthetic_frames = []
for _ in range(N_FRAMES):
    frame = np.full((HEIGHT, WIDTH, 3), 210, dtype=np.uint8)
    cv2.rectangle(frame, (0, HEIGHT - 60), (WIDTH, HEIGHT), (120, 120, 120), -1)
    for obj in objects:
        obj["x"] = int(obj["x"] + obj["vx"])
        obj["y"] = int(obj["y"] + obj["vy"])
        if obj["x"] <= 0 or obj["x"] + obj["w"] >= WIDTH:  obj["vx"] *= -1
        if obj["y"] <= 0 or obj["y"] + obj["h"] >= HEIGHT - 60: obj["vy"] *= -1
        obj["x"] = max(0, min(obj["x"], WIDTH - obj["w"]))
        obj["y"] = max(0, min(obj["y"], HEIGHT - obj["h"]))
        cv2.rectangle(frame, (obj["x"], obj["y"]),
                      (obj["x"]+obj["w"], obj["y"]+obj["h"]), obj["color"], -1)
    synthetic_frames.append(frame)

print(f"Generated {len(synthetic_frames)} frames  ({WIDTH}×{HEIGHT} @ {FPS} FPS)")


In [ ]:
# ── Run V1 pipeline ───────────────────────────────────────────────────────────
WARMUP = 20
detector = MOG2Detector(min_area=500)
tracker  = CentroidTracker()

detector.warmup(synthetic_frames[:WARMUP])

annotated_v1 = []
for i, frame in enumerate(synthetic_frames):
    dets   = detector.detect(frame)
    tracks = tracker.update(dets)
    annotated_v1.append(draw_tracks(frame, tracks))

print(f"Processed {len(annotated_v1)} frames")


In [ ]:
# ── Display first 5 annotated frames ─────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
sample_indices = [20, 40, 60, 90, 130]
for ax, idx in zip(axes, sample_indices):
    rgb = cv2.cvtColor(annotated_v1[idx], cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.set_title(f"Frame {idx}", fontsize=11)
    ax.axis("off")
plt.suptitle("V1: MOG2 + Centroid Tracker — Annotated Frames", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../report/figures/v1_notebook_sample.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Speed benchmark ───────────────────────────────────────────────────────────
def v1_step(frame):
    return tracker.update(detector.detect(frame))

speed = benchmark_speed(v1_step, synthetic_frames[60], n_runs=10)
print(f"V1 Speed: {speed['fps']:.1f} FPS  |  {speed['mean_ms']:.1f} ± {speed['std_ms']:.1f} ms/frame")


In [ ]:
# ── Save output video ─────────────────────────────────────────────────────────
os.makedirs("../outputs", exist_ok=True)
write_output_video(annotated_v1, "../outputs/v1_output.mp4", FPS)
print("Saved: ../outputs/v1_output.mp4")


## 2. Running on Modal

```bash
# Run V1 pipeline on Modal (CPU worker)
modal run modal_app/modal_v1_scratch.py

# Run both V1 and V2 together
modal run modal_app/modal_run_all.py
```


In [ ]:
# ── Compute metrics ───────────────────────────────────────────────────────────
from src.utils import evaluate_tracking

# Build pseudo ground-truth from object positions
gt = [{"bbox": [o["x"], o["y"], o["x"]+o["w"], o["y"]+o["h"]], "id": i}
      for i, o in enumerate(objects)]

# Get last frame predictions
last_tracks = tracker.tracks
pred = [{"bbox": t["bbox"], "id": t["id"]} for t in last_tracks.values()]

metrics = evaluate_tracking(gt, pred)
print("Tracking Metrics (approx, last frame):")
for k, v in metrics.items():
    print(f"  {k:15s}: {v:.4f}" if isinstance(v, float) else f"  {k:15s}: {v}")


## 3. Observations

**Strengths of V1:**
- No pre-trained weights required — zero internet dependency at inference time
- Very fast on CPU (~40–50 FPS on modern hardware) due to highly-optimised OpenCV routines
- Fully interpretable — every step can be tuned with explicit parameters

**Limitations:**
- Sensitive to illumination changes and camera motion
- Foreground blobs merge when persons overlap, causing ID switches
- Requires a static background or warmup period to build the background model
- Proxy confidence (area-based) is not semantically meaningful

**When to use V1:** Fixed cameras, edge devices, controlled environments, privacy-sensitive deployments.
